In [1]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from PIL import Image
import os, random
import pandas as pd

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.colors import to_rgba
import seaborn as sns
sns.set_theme('notebook', style='whitegrid')

## Progress bar
from tqdm.auto import tqdm

In [2]:
import torch
torch.manual_seed(42) # Setting the seed

print("Using torch", torch.__version__)

Using torch 2.10.0+cu128


In [3]:
df =  pd.read_csv("dataset/toy_dataset_label.csv", sep="\t" ,on_bad_lines="skip")

print(df.head(8))

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/toy_dataset_label.csv'

In [4]:
techniques = pd.unique(df['TECHNIQUE'])

print(len(list(techniques)))

21437


In [5]:
df['TECHNIQUE_CLEAN'] = df['TECHNIQUE'].str.split(',').str[0]

df['TECHNIQUE_CLEAN'] = df['TECHNIQUE_CLEAN'].str.strip().str.lower()

print(len(pd.unique(df['TECHNIQUE_CLEAN'])))

print(f"unique classes: {list(pd.unique(df['TECHNIQUE_CLEAN']))}")

2604
unique classes: ['oil on copper', 'oil on canvas', 'copperplate', 'wood', 'maiolica', 'ceramics', 'ceramic mural composition', '-', 'marble', 'oil on panel', 'oil on cardboard', 'silver', 'engraving', 'wax on obsidian', 'oil and gouache on paper', 'bronze', 'lead', 'drawing', 'terracotta', 'photo', 'white marble', 'fresco', 'oil on oak panel', 'brass', 'cast copper alloy', 'black chalk on vellum', 'stained glass', 'oil on wood', 'oil on oak', 'oil on oak wood', 'oil on paper laid down on board', 'gilt and painted terracotta', 'terracotta with traces of paint', 'glazed terracotta', 'pietra caciolfa', 'pencil and gouache on paper', 'tempera and leaf on panel', 'tempera on panel', 'oil on wood transferred to canvas', 'black crayon with white highlights on blue paper', 'engraving. 320 x 170 cm', 'engraving. 280 x 190 mm', 'panel', 'watercolour', 'woodcut', 'tempera and gold on wood', 'tempera on wood', 'stucco', 'black marble', 'oil on canvas : 118 x 92 cm', 'oil on canvas : 86 x 136 

In [6]:
# Filter techniques with count >= 50
technique_counts = df['TECHNIQUE_CLEAN'].value_counts()
techniques_filtered = technique_counts[technique_counts >= 50].index.tolist()

# Remove '-' if it exists
techniques_filtered = [t for t in techniques_filtered if t != '-']

print(f"Techniques with count >= 50: {techniques_filtered}")
print(f"Number of techniques: {len(techniques_filtered)}")

# Filter dataframe to keep only these techniques
df_filtered = df[df['TECHNIQUE_CLEAN'].isin(techniques_filtered)]

# === Riduci dimensione del dataset per training più veloce (seleziona valore) ===
# Imposta la frazione dei campioni da utilizzare; usa 1.0 per tutto il dataset
# Valori più piccoli accelerano il training ma possono aumentare l'overfitting
fraction = 0.1  # default: usa il 10% dei campioni

df_filtered = df_filtered.sample(frac=fraction, random_state=42).reset_index(drop=True)
print(f"After downsampling, dataset size: {len(df_filtered)} ({fraction*100:.1f}%)")

print(f"\nOriginal dataset size: {len(df)}")

print(f"Filtered dataset size: {len(df_filtered)}")
print(f"\nFiltered technique counts:\n{df_filtered['TECHNIQUE_CLEAN'].value_counts()}")

Techniques with count >= 50: ['oil on canvas', 'fresco', 'oil on panel', 'photo', 'oil on wood', 'marble', 'tempera on wood', 'bronze', 'oil on oak panel', 'tempera on panel', 'engraving', 'stone', 'panel', 'wood', 'oil on copper', 'woodcut', 'oil on oak', 'etching', 'mosaic', 'manuscript', 'tempera on canvas', 'pen', 'black chalk', 'egg tempera on wood', 'drawing', 'terracotta', 'pen and brown ink', 'tempera and gold on panel', 'glazed terracotta', 'oak', 'stained glass window', 'oil on cardboard', 'watercolour', 'red chalk', 'tempera and gold on parchment', 'white marble', 'painted wood', 'silver', 'pastel on paper', 'pencil', 'oil on poplar panel', 'polychrome wood', 'pen and ink on paper']
Number of techniques: 43
After downsampling, dataset size: 3641 (10.0%)

Original dataset size: 43455
Filtered dataset size: 3641

Filtered technique counts:
TECHNIQUE_CLEAN
oil on canvas                    1321
fresco                            468
oil on panel                      315
photo    

In [ ]:
def group_techniques_final(tech):
    tech = str(tech).lower()
    
    # 1. Pittura (Priorità alle tecniche più comuni)
    if 'oil' in tech: return 'oil painting'
    if 'fresco' in tech: return 'fresco'
    if 'tempera' in tech: return 'tempera'
    if 'watercol' in tech or 'water-col' in tech: return 'watercolor'
    
    # 2. Grafica e Disegno (Unifichiamo engraving, etching, woodcut e chalk)
    if any(x in tech for x in ['engraving', 'etching', 'woodcut']): return 'engraving'
    if any(x in tech for x in ['drawing', 'ink', 'chalk', 'pen', 'graphite', 'pencil']): return 'drawing'
    
    # 3. Scultura e Materiali Rigidi
    if any(x in tech for x in ['marble', 'stone', 'terracotta', 'clay', 'glazed']): return 'sculpture stone'
    if any(x in tech for x in ['bronze', 'metal', 'gold', 'silver', 'copper']): return 'sculpture metal'
    if any(x in tech for x in ['wood', 'panel', 'oak']): 
    # Se nel testo c'è anche "oil" o "tempera", probabilmente è un dipinto, non una scultura
        if any(p in tech for p in ['oil', 'tempera', 'acrylic']):
            return 'pittura_su_tavola'
        return 'wood sculpture'
    
    # 4. Altro (Vetro, Mosaici, Miniature)
    if 'mosaic' in tech: return 'mosaic'
    if 'glass' in tech: return 'glass art'
    if 'manuscript' in tech or 'illumination' in tech: return 'manuscript'
    if 'photo' in tech: return 'photography'
    
    return None

df['TECHNIQUE_GROUPED'] = df['TECHNIQUE_CLEAN'].apply(group_techniques_final)
df_filtered['TECHNIQUE_GROUPED'] = df_filtered['TECHNIQUE_CLEAN'].apply(group_techniques_final)

# Remove rows classified as None (unrecognized techniques)
df = df[df['TECHNIQUE_GROUPED'].notna()]
df_filtered = df_filtered[df_filtered['TECHNIQUE_GROUPED'].notna()]

In [ ]:
print(f"Total classes: {len(pd.unique(df_filtered['TECHNIQUE_GROUPED']))}")
print(f"Classes: {sorted(list(pd.unique(df_filtered['TECHNIQUE_GROUPED'])))}")
print(f"\nClass distribution:\n{df_filtered['TECHNIQUE_GROUPED'].value_counts()}")

<StringArray>
[   'oil painting', 'sculpture metal',  'wood sculpture',           'other',
 'sculpture stone',       'engraving',         'drawing',     'photography',
          'fresco',       'glass art',         'tempera',      'watercolor',
          'mosaic',      'manuscript']
Length: 14, dtype: str
TECHNIQUE_GROUPED
oil painting       21295
fresco              4477
sculpture stone     3052
tempera             2646
photography         2060
drawing             1865
wood sculpture      1690
manuscript          1435
sculpture metal     1430
engraving           1425
other               1400
watercolor           309
mosaic               233
glass art            138
Name: count, dtype: int64


In [23]:
# === Handling Class Imbalance ===

# 1. Calcola class weights (inverse della frequenza)
class_counts = df_filtered['TECHNIQUE_GROUPED'].value_counts()
total_samples = len(df_filtered)
class_weights = {}

for technique in df_filtered['TECHNIQUE_GROUPED'].unique():
    count = class_counts.get(technique, 0)
    weight = total_samples / (len(df_filtered['TECHNIQUE_GROUPED'].unique()) * count) if count > 0 else 0
    class_weights[technique] = weight

print("Class Weights (higher = less frequent class):")
for technique, weight in sorted(class_weights.items(), key=lambda x: x[1], reverse=True):
    print(f"  {technique}: {weight:.4f}")

# 2. Crea mapping da label a indice
techniques_grouped = sorted(df_filtered['TECHNIQUE_GROUPED'].unique())
class_to_idx = {technique: idx for idx, technique in enumerate(techniques_grouped)}
idx_to_class = {idx: technique for technique, idx in class_to_idx.items()}

print("\nClass to Index mapping:")
print(class_to_idx)

# 3. Per usare i weights nel training con PyTorch:
# Opzione A: Usare WeightedRandomSampler nel DataLoader
weights_list = [class_weights[df_filtered.iloc[i]['TECHNIQUE_GROUPED']] 
                 for i in range(len(df_filtered))]

print(f"\n✓ Use WeightedRandomSampler in DataLoader:")
print(f"  sampler = WeightedRandomSampler(weights_list, num_samples=len(df_filtered), replacement=True)")
print(f"  dataloader = DataLoader(dataset, sampler=sampler, batch_size=32)")

# Opzione B: Usare weights nella loss function (CrossEntropyLoss)
class_weights_tensor = torch.tensor([class_weights[t] for t in techniques_grouped], dtype=torch.float32)
print(f"\n✓ Use class weights in CrossEntropyLoss:")
print(f"  criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)")
print(f"  class_weights_tensor: {class_weights_tensor}")

Class Weights (higher = less frequent class):
  other: 43.3452
  glass art: 32.5089
  watercolor: 32.5089
  manuscript: 10.0027
  mosaic: 10.0027
  sculpture metal: 3.1716
  drawing: 2.9554
  wood sculpture: 2.6811
  engraving: 2.0005
  tempera: 1.2210
  photography: 1.1259
  sculpture stone: 0.9632
  fresco: 0.5557
  oil painting: 0.1308

Class to Index mapping:
{'drawing': 0, 'engraving': 1, 'fresco': 2, 'glass art': 3, 'manuscript': 4, 'mosaic': 5, 'oil painting': 6, 'other': 7, 'photography': 8, 'sculpture metal': 9, 'sculpture stone': 10, 'tempera': 11, 'watercolor': 12, 'wood sculpture': 13}

✓ Use WeightedRandomSampler in DataLoader:
  sampler = WeightedRandomSampler(weights_list, num_samples=len(df_filtered), replacement=True)
  dataloader = DataLoader(dataset, sampler=sampler, batch_size=32)

✓ Use class weights in CrossEntropyLoss:
  criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
  class_weights_tensor: tensor([ 2.9554,  2.0005,  0.5557, 32.5089, 10.0027, 10.002

In [ ]:
# === Custom Dataset Class (adattato al tuo codice) ===
class CustomImageDataset(Dataset):
    def __init__(self, dataframe, class_to_idx, img_dir='dataset/toy_dataset', transform=None):
        """
        Args:
            dataframe: DataFrame con colonne 'FILE' e 'TECHNIQUE_GROUPED'
            class_to_idx: dict mapping classe a indice numerico
            img_dir: directory delle immagini
            transform: trasformazioni da applicare
        """
        self.dataframe = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        # Ottieni il path dell'immagine dalla colonna 'FILE'
        img_filename = self.dataframe.iloc[index]['FILE']
        img_path = os.path.join(self.img_dir, img_filename)

        # Carica l'immagine
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Errore caricamento immagine {img_path}: {e}")
            # Fallback: immagine vuota
            image = Image.new('RGB', (224, 224))

        # Ottieni il label dalla colonna 'TECHNIQUE_GROUPED'
        label_str = self.dataframe.iloc[index]['TECHNIQUE_GROUPED']
        # Converti string label a integer
        label = self.class_to_idx[label_str]

        if self.transform:
            image = self.transform(image)

        return image, label

# === Trasformazioni per DenseNet-121 con Data Augmentation ===

# DenseNet-121 richiede input 224x224
# Aggiungiamo data augmentation per ridurre overfitting
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize iniziale più grande
    transforms.RandomCrop(224),     # Crop casuale a 224x224
    transforms.RandomHorizontalFlip(),  # Flip orizzontale casuale
    transforms.RandomRotation(15),   # Rotazione casuale fino a 15 gradi
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

transform_val = transforms.Compose([
    transforms.Resize(256),        # Ridimensiona il lato corto a 256
    transforms.CenterCrop(224),   # Ritaglia il centro senza distorcere
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# === Crea i dataset ===

# Split: 80% train, 20% val
train_size = int(0.8 * len(df_filtered))
val_size = len(df_filtered) - train_size
train_df, val_df = random_split(df_filtered, [train_size, val_size])

# Converte in DataFrame
train_indices = train_df.indices
val_indices = val_df.indices
train_df_split = df_filtered.iloc[train_indices].reset_index(drop=True)
val_df_split = df_filtered.iloc[val_indices].reset_index(drop=True)

# Crea i dataset (applica trasformazioni separate per train/val)
train_dataset = CustomImageDataset(
    dataframe=train_df_split,
    class_to_idx=class_to_idx,
    img_dir='dataset/toy_dataset',
    transform=transform_train
)

val_dataset = CustomImageDataset(
    dataframe=val_df_split,
    class_to_idx=class_to_idx,
    img_dir='dataset/toy_dataset',
    transform=transform_val
)

# === Crea i DataLoader ===

batch_size = 32

# Per il training usa WeightedRandomSampler per bilanciare le classi
counts = train_df_split['TECHNIQUE_GROUPED'].value_counts()
weights_per_class = {cls: 1.0/count for cls, count in counts.items()}
train_weights = [weights_per_class[label] for label in train_df_split['TECHNIQUE_GROUPED']]
train_sampler = WeightedRandomSampler(train_weights, num_samples=len(train_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# === Test dei DataLoader ===

# Esempio: Itera attraverso i dati di training
for images, labels in train_loader:
    print(f"Train - images shape: {images.shape}, labels shape: {labels.shape}")
    print(f"Labels: {labels[:10]}")  # Mostra primi 10 label
    break

# Esempio: Itera attraverso i dati di validation
for images, labels in val_loader:
    print(f"Validation - images shape: {images.shape}, labels shape: {labels.shape}")
    print(f"Labels: {labels[:10]}")  # Mostra primi 10 label
    break

Train dataset size: 2912
Validation dataset size: 729
Train batches per epoch: 91
Validation batches: 23


KeyError: 'black chalk'

In [20]:
# === Visualizza alcune immagini con le loro true label ===

# Funzione semplificata per mostrare un'immagine
def plot_example_image(dataset, index):
    image, label = dataset[index]
    
    # Denormalizza l'immagine (semplice come nell'esempio)
    image = image * torch.tensor([0.229, 0.224, 0.225]).unsqueeze(1).unsqueeze(2) + torch.tensor([0.485, 0.456, 0.406]).unsqueeze(1).unsqueeze(2)
    
    # Converti in numpy array per il plotting
    image = image.permute(1, 2, 0).numpy()  # Da CxHxW a HxWxC
    image = image.clip(0, 1)  # Assicura valori tra 0 e 1

    # Ottieni la stringa del label
    label_str = idx_to_class[label]  # Usa il mapping globale

    # Mostra l'immagine
    plt.imshow(image)
    plt.title(f"Label: {label_str}")
    plt.axis('off')
    plt.show()

# Mostra alcune immagini di esempio
plot_example_image(train_dataset, index=3)

KeyError: 14

In [21]:
# === Implementazione di DenseNet-121 per la classificazione delle tecniche artistiche ===

# Carica il modello pre-addestrato DenseNet-121 con pesi ImageNet
model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False

# Sblocchiamo l'ultimo blocco denso (il 4) e lo strato di normalizzazione finale
for param in model.features.denseblock4.parameters():
    param.requires_grad = True
for param in model.features.norm5.parameters():
    param.requires_grad = True

# Aggiungi un classificatore con dropout per regolarizzazione
num_features = model.classifier.in_features
num_classes = len(class_to_idx)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(num_features, num_classes)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001,
    weight_decay=1e-4
)

# Funzione di addestramento del modello con early stopping

def train_model(model, train_loader, val_loader, criterion, optimizer,
                num_epochs=20, patience=5):
    best_val_acc = 0.0
    epochs_no_improve = 0
    best_model_state = None

    for epoch in range(num_epochs):
        # training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training", leave=True)
        for images, labels in train_pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            current_loss = running_loss / (total // batch_size + 1)
            current_acc = 100 * correct / total
            train_pbar.set_postfix({'Loss': f'{current_loss:.4f}', 'Acc': f'{current_acc:.2f}%'} )

        train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct / total

        # validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation", leave=False)
            for images, labels in val_pbar:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                current_val_loss = val_loss / (val_total // batch_size + 1)
                current_val_acc = 100 * val_correct / val_total
                val_pbar.set_postfix({'Val Loss': f'{current_val_loss:.4f}', 'Val Acc': f'{current_val_acc:.2f}%'} )

        val_loss /= len(val_loader)
        val_accuracy = 100 * val_correct / val_total

        print(f"Epoch [{epoch+1}/{num_epochs}], "
              f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, "
              f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            epochs_no_improve = 0
            best_model_state = model.state_dict()
            torch.save(best_model_state, 'model/best_densenet121.tar')
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"No improvement for {patience} epochs. Early stopping.")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Loaded best model with val accuracy {best_val_acc:.2f}%")
    return model

# Lancio dell'addestramento (early stopping incluso)
model = train_model(model, train_loader, val_loader, criterion, optimizer)


Epoch 1/20 - Training:  87%|████████▋ | 79/91 [06:16<00:57,  4.76s/it, Loss=2.5707, Acc=16.02%]


KeyboardInterrupt: 